## SMR by State

This notebook follows the `merged_df_table` construction used in `02_data_preparation.ipynb`:
- incident counts come from `incident`
- enrollment comes from `enrollment_state_year_mat`
- state-year rows are built from enrollment, with missing incident counts filled as 0


In [2]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from supabase import create_client

project_root = Path.cwd().parent
env_path = project_root / ".env"

load_dotenv(dotenv_path=env_path, override=True)

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

if not SUPABASE_URL or not SUPABASE_KEY:
    raise ValueError(f"Missing SUPABASE_URL or SUPABASE_KEY in {env_path}")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

print("Supabase connection configured.")


Supabase connection configured.


In [3]:
def fetch_all_rows(table_name, columns="*", order_column=None, page_size=1000):
    all_rows = []
    start = 0

    while True:
        query = supabase.table(table_name).select(columns)
        if order_column is not None:
            query = query.order(order_column)

        response = query.range(start, start + page_size - 1).execute()
        data = response.data or []

        if not data:
            break

        all_rows.extend(data)
        start += page_size

    return all_rows


In [4]:
state_abbrev = {
    "ALABAMA": "AL", "ALASKA": "AK", "ARIZONA": "AZ", "ARKANSAS": "AR",
    "CALIFORNIA": "CA", "COLORADO": "CO", "CONNECTICUT": "CT",
    "DELAWARE": "DE", "DISTRICT OF COLUMBIA": "DC", "FLORIDA": "FL",
    "GEORGIA": "GA", "HAWAII": "HI", "IDAHO": "ID", "ILLINOIS": "IL",
    "INDIANA": "IN", "IOWA": "IA", "KANSAS": "KS", "KENTUCKY": "KY",
    "LOUISIANA": "LA", "MAINE": "ME", "MARYLAND": "MD",
    "MASSACHUSETTS": "MA", "MICHIGAN": "MI", "MINNESOTA": "MN",
    "MISSISSIPPI": "MS", "MISSOURI": "MO", "MONTANA": "MT",
    "NEBRASKA": "NE", "NEVADA": "NV", "NEW HAMPSHIRE": "NH",
    "NEW JERSEY": "NJ", "NEW MEXICO": "NM", "NEW YORK": "NY",
    "NORTH CAROLINA": "NC", "NORTH DAKOTA": "ND", "OHIO": "OH",
    "OKLAHOMA": "OK", "OREGON": "OR", "PENNSYLVANIA": "PA",
    "RHODE ISLAND": "RI", "SOUTH CAROLINA": "SC",
    "SOUTH DAKOTA": "SD", "TENNESSEE": "TN", "TEXAS": "TX",
    "UTAH": "UT", "VERMONT": "VT", "VIRGINIA": "VA",
    "WASHINGTON": "WA", "WEST VIRGINIA": "WV",
    "WISCONSIN": "WI", "WYOMING": "WY"
}

valid_states = set(state_abbrev.values())

incident_rows = fetch_all_rows(
    table_name="incident",
    columns="Incident_ID,State,Year",
    order_column="Incident_ID",
)
incident_df = pd.DataFrame(incident_rows)
incident_df["Year"] = pd.to_numeric(incident_df["Year"], errors="coerce")

incident_df = incident_df[
    incident_df["State"].isin(valid_states) & incident_df["Year"].ge(1987)
].copy()
incident_df["Year"] = incident_df["Year"].astype(int)

incident_agg = (
    incident_df.groupby(["State", "Year"], as_index=False)
    .size()
    .rename(columns={"size": "incident_count"})
)

print(f"Incident state-year rows: {len(incident_agg)}")


Incident state-year rows: 861


In [5]:
enrollment_rows = fetch_all_rows(
    table_name="enrollment_state_year_mat",
    columns="state,year,total_students",
)
enrollment_df = pd.DataFrame(enrollment_rows)

enrollment_df["state"] = enrollment_df["state"].astype(str).str.upper()
enrollment_df["State"] = enrollment_df["state"].map(state_abbrev)
enrollment_df["Year"] = pd.to_numeric(enrollment_df["year"], errors="coerce")
enrollment_df["total_students"] = pd.to_numeric(enrollment_df["total_students"], errors="coerce")

enrollment_df = enrollment_df.dropna(subset=["State", "Year", "total_students"]).copy()
enrollment_df["Year"] = enrollment_df["Year"].astype(int)

enrollment_df = (
    enrollment_df.sort_values("total_students", ascending=False)
    .drop_duplicates(["State", "Year"])
)

print(f"Enrollment state-year rows: {len(enrollment_df)}")


Enrollment state-year rows: 1989


In [6]:
merged_df_table = enrollment_df[["State", "Year", "total_students"]].merge(
    incident_agg,
    on=["State", "Year"],
    how="left",
)

merged_df_table["incident_count"] = merged_df_table["incident_count"].fillna(0).astype(int)

print(merged_df_table.shape)
print(f"Zero-incident state-year rows: {(merged_df_table['incident_count'] == 0).sum()}")
merged_df_table.head()


(1989, 4)
Zero-incident state-year rows: 1128


,State,Year,total_students,incident_count
0,CA,2005,6322586,6
1,CA,2006,6312103,4
2,CA,2004,6298928,7
3,CA,2007,6274813,5
4,CA,2003,6244403,4


In [7]:
state_smr = (
    merged_df_table.groupby("State", as_index=False)
    .agg(
        total_incidents=("incident_count", "sum"),
        total_enrollment=("total_students", "sum"),
    )
    .rename(columns={"State": "state"})
)

total_incidents_all = state_smr["total_incidents"].sum()
total_enrollment_all = state_smr["total_enrollment"].sum()
national_rate = total_incidents_all / total_enrollment_all if total_enrollment_all else pd.NA

state_smr["expected"] = state_smr["total_enrollment"] * national_rate
state_smr["smr"] = state_smr["total_incidents"] / state_smr["expected"].where(state_smr["expected"].ne(0))
state_smr["log_smr"] = np.log(state_smr["smr"].where(state_smr["smr"].gt(0)))
state_smr = state_smr[["state", "total_incidents", "expected", "smr", "log_smr"]].sort_values("state").reset_index(drop=True)

output_path = project_root / "outputs" / "state_smr_from_supabase.csv"
state_smr.to_csv(output_path, index=False)

print(f"National rate per student: {national_rate}")
print(f"Saved results to: {output_path}")
state_smr


National rate per student: 1.536391513671252e-06
Saved results to: /home/tommie-clark/capstoneProje t/capstoneProject2026/outputs/state_smr_from_supabase.csv


,state,total_incidents,expected,smr,log_smr
0,AK,6,7.510066,0.798928,-0.224485
1,AL,66,43.234925,1.526544,0.423006
2,AR,32,27.158628,1.178263,0.164041
3,AZ,29,54.926448,0.527979,-0.638699
4,CA,249,338.158926,0.736340,-0.306063
5,CO,46,44.559567,1.032326,0.031815
6,CT,23,31.017881,0.741508,-0.299070
7,DC,48,4.572057,10.498557,2.351238
8,DE,21,7.114565,2.951691,1.082378
9,FL,139,144.847893,0.959627,-0.041210


In [8]:
heatmap_output_path = project_root / "outputs" / "state_smr_heatmap.html"

fig = px.choropleth(
    state_smr,
    locations="state",
    locationmode="USA-states",
    scope="usa",
    color="log_smr",
    color_continuous_scale="RdYlBu_r",
    color_continuous_midpoint=0,
    hover_name="state",
    hover_data={
        "total_incidents": ":,.0f",
        "expected": ":,.2f",
        "smr": ":.3f",
        "log_smr": ":.3f",
    },
    labels={
        "log_smr": "log(SMR)",
        "total_incidents": "Observed",
        "expected": "Expected",
        "state": "State",
    },
    title="log(SMR) by State",
)

fig.update_layout(
    margin={"l": 0, "r": 0, "t": 60, "b": 0},
    coloraxis_colorbar={"title": "log(SMR)"},
)

fig.write_html(heatmap_output_path)
print(f"Saved heatmap to: {heatmap_output_path}")
fig


Saved heatmap to: /home/tommie-clark/capstoneProje t/capstoneProject2026/outputs/state_smr_heatmap.html
